# v13.2.1 full diagnostics

This notebook runs the v13.2.1 diagnostic release and previews the key W/H/D recovery files.

Use `MODE = "quick"` for a smoke test. Use `MODE = "wide"` for the full diagnostic run.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
SCRIPT = ROOT / "v13_2_1_full_diagnostics_block_discovery.py"

# Change to "wide" for the complete run.
MODE = "quick"
OUTDIR = ROOT / f"v13_2_1_{MODE}_outputs"

print("ROOT:", ROOT)
print("SCRIPT:", SCRIPT)
print("MODE:", MODE)
print("OUTDIR:", OUTDIR)

In [ ]:
cmd = [sys.executable, str(SCRIPT), f"--{MODE}", "--outdir", str(OUTDIR)]
print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"v13.2.1 run failed with code {result.returncode}")

print("Done:", OUTDIR)

## Selected model summary

In [ ]:
report_path = OUTDIR / "v13_2_1_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))

selected = pd.Series(report["selected"], name="value")
display(selected.to_frame())

posterior_summary = report.get("posterior_summary", {})
display(pd.Series({
    "min_inclusion_prob": posterior_summary.get("min_inclusion_prob"),
    "mean_inclusion_prob": posterior_summary.get("mean_inclusion_prob"),
    "residual_noise_var": posterior_summary.get("residual_noise_var"),
    "posterior_pruned": posterior_summary.get("posterior_pruned"),
}, name="value").to_frame())

## Full space diagnostics

In [ ]:
space = pd.read_csv(OUTDIR / "v13_2_1_space_diagnostics.csv")
display(space)

watch = [
    "W_raw_subspace_angle_mean_deg",
    "W_raw_subspace_angle_max_deg",
    "H_raw_subspace_angle_mean_deg",
    "H_raw_subspace_angle_max_deg",
    "center_match_mae",
    "component_alignment_score_mean",
    "coassociation_auc_like",
    "D_localized_auc_like",
    "D_true_auc_like",
]
space_watch = space[space["metric"].isin(watch)].copy()
display(space_watch)

## Detailed tables

In [ ]:
for name in [
    "v13_2_1_subspace_angles.csv",
    "v13_2_1_center_matching.csv",
    "v13_2_1_component_alignment.csv",
    "v13_2_1_block_posterior_summary.csv",
]:
    print("\n===", name, "===")
    display(pd.read_csv(OUTDIR / name))

## Main plots

In [ ]:
plots = [
    "v13_2_1_iterative_K_trace.png",
    "v13_2_1_iterative_gain_trace.png",
    "v13_2_1_component_inclusion_posterior.png",
    "v13_2_1_F_posterior_mass.png",
    "v13_2_1_block_coassociation_heatmap.png",
    "v13_2_1_localized_component_centers.png",
]

for plot in plots:
    path = OUTDIR / plot
    if path.exists():
        display(Markdown(f"### {plot}"))
        display(Image(filename=str(path)))

## Raw matrices

`v13_2_1_latent_matrices.npz` contains raw arrays for follow-up analysis: W/H raw, localized, true matrices, D matrices, coassociation matrix, centers, labels, and selected assignment.

In [ ]:
import numpy as np

npz_path = OUTDIR / "v13_2_1_latent_matrices.npz"
with np.load(npz_path, allow_pickle=True) as data:
    print("Arrays in", npz_path.name)
    for key in data.files:
        arr = data[key]
        print(f"{key:28s}", arr.shape, arr.dtype)